In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvamFuZHUvcmVwb3MvTkJWL2RvY3MvY29udGVudHMvZXh0LWltcGw='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/importlib/_bootstrap.py": 1761174496.0735826, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/importlib/_bootstrap_external.py": 1761174496.0715826, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/zipimport.py": 1761174495.2065685, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/codecs.py": 1761174494.8845632, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/encodings/aliases.py": 1761174495.553574, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/encodings/__init__.py": 1761174495.666576, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/encodings/utf_8.py": 1761174495.789578, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/abc.py": 1761174494.8435626, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/io.py": 1761174494.9785647, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/stat.py": 1761174495.124567, "/home/jandu/miniforge3/envs/aria-nbv/lib/python3.11/_collections_abc.py": 1761174494.8905

In [2]:
# | echo: false
# | output: false
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parents[2]

# Define data path
DATA_PATH = repo_root / ".data" / "semidense_samples" / "ase" / "ase_examples" / "0"
assert DATA_PATH.exists(), f"Data path not found: {DATA_PATH}"
print(f"Using data from: {DATA_PATH}")

Using data from: /home/jandu/repos/NBV/.data/semidense_samples/ase/ase_examples/0


In [3]:
#| eval: false
#| echo: true
# Note: VRS data provider requires actual VRS recording files
# For ASE dataset, we load images directly (see section 5.2)
from projectaria_tools.core import data_provider
from projectaria_tools.core.stream_id import StreamId
from projectaria_tools.core.sensor_data import TimeDomain, TimeQueryOptions

# Example usage (requires VRS file):
# provider = data_provider.create_vrs_data_provider("path/to/recording.vrs")
# print(f"Provider type: {type(provider)}")

# Get stream IDs
# rgb_stream_id = StreamId("214-1")
# slam_stream_id = provider.get_stream_id_from_label("camera-slam-left")
# print(f"RGB Stream ID: {rgb_stream_id}")
# print(f"SLAM Stream ID: {slam_stream_id}")

# Query by index
# image_data, record = provider.get_image_data_by_index(rgb_stream_id, frame_idx=0)
# image_array = image_data.to_numpy_array()
# print(f"Image shape: {image_array.shape}, dtype: {image_array.dtype}")
# print(f"Record timestamp: {record.capture_timestamp_ns} ns")

print("VRS provider section - requires VRS recording file")
print("See section 5.2 for ASE dataset loading with actual data")

In [4]:
#| echo: true
from projectaria_tools.projects.ase import readers
import numpy as np

# Load camera trajectory (6DoF poses)
trajectory = readers.read_trajectory_file(str(DATA_PATH / "trajectory.csv"))
Ts_world_device = trajectory["Ts_world_from_device"]  # List[SE3]
timestamps = trajectory["timestamps"]  # np.ndarray
print(f"Trajectory keys: {list(trajectory.keys())}")
print(f"Number of poses: {len(Ts_world_device)}")
print(f"Timestamps shape: {timestamps.shape}, dtype: {timestamps.dtype}")
print(f"First pose (4x4 matrix):\n{Ts_world_device[0].to_matrix()}")

# Load semi-dense point cloud
points = readers.read_points_file(str(DATA_PATH / "semidense_points.csv.gz"))
print(f"\nPoints shape: {points.shape}, dtype: {points.dtype}")
print(f"Point coordinates range:")
print(f"  X: [{points[:, 0].min():.2f}, {points[:, 0].max():.2f}]")
print(f"  Y: [{points[:, 1].min():.2f}, {points[:, 1].max():.2f}]")
print(f"  Z: [{points[:, 2].min():.2f}, {points[:, 2].max():.2f}]")

# Load SceneScript language (ground truth entities)
entities = readers.read_language_file(str(DATA_PATH / "ase_scene_language.txt"))
print(f"\nNumber of entities: {len(entities)}")
print(f"First 3 entities:")
for i, (cmd, params) in enumerate(entities[:3]):
    print(f"  {i}: {cmd} - {list(params.keys())[:5]}")

Loaded trajectory with 350 device poses.
Trajectory keys: ['Ts_world_from_device', 'timestamps']
Number of poses: 350
Timestamps shape: (350,), dtype: int64
First pose (4x4 matrix):
[[ 0.00534871  0.62566658  0.78007226  1.51215362]
 [-0.17628863  0.76845577 -0.61514067  3.88807193]
 [-0.98432399 -0.13422766  0.11440815  1.43420936]
 [ 0.          0.          0.          1.        ]]


Loaded #3dPoints: 433426



Points shape: (433426, 3), dtype: float64
Point coordinates range:
  X: [-12.49, 9.05]
  Y: [-6.08, 8.62]
  Z: [-5.50, 5.42]
Loaded scene commands with a total of 15 entities.

Number of entities: 15
First 3 entities:
  0: make_wall - ['id', 'a_x', 'a_y', 'a_z', 'b_x']
  1: make_wall - ['id', 'a_x', 'a_y', 'a_z', 'b_x']
  2: make_wall - ['id', 'a_x', 'a_y', 'a_z', 'b_x']


In [5]:
#| echo: true
from projectaria_tools.projects.ase.interpreter import language_to_bboxes

# Convert entities to 3D bounding boxes
entity_boxes = language_to_bboxes(entities)
print(f"Number of bounding boxes: {len(entity_boxes)}")

# Each box contains:
# - 'class': object category
# - 'center': (3,) center position
# - 'scale': (3,) dimensions
# - 'rotation': (3, 3) rotation matrix
if len(entity_boxes) > 0:
    print(f"\nFirst bounding box:")
    box = entity_boxes[0]
    print(f"  Class: {box['class']}")
    print(f"  Center: {box['center']}")
    print(f"  Scale: {box['scale']}")
    print(f"  Rotation shape: {box['rotation'].shape}")

    # Show statistics across all boxes
    classes = [b['class'] for b in entity_boxes]
    from collections import Counter
    print(f"\nObject class distribution:")
    for cls, count in Counter(classes).most_common(5):
        print(f"  {cls}: {count}")

Scene contains:
  wall: 8
  door: 3
  window: 4
Number of bounding boxes: 15

First bounding box:
  Class: wall
  Center: [1.25197755 6.16466464 1.63121486]
  Scale: [7.63439633 0.         3.26242971]
  Rotation shape: (3, 3)

Object class distribution:
  wall: 8
  window: 4
  door: 3


In [6]:
#| eval: false
#| echo: true
from projectaria_tools.core import mps
from projectaria_tools.core.mps.utils import filter_points_from_confidence

# Load global point cloud with confidence
points = mps.read_global_point_cloud("global_points.csv.gz")
print(f"Points shape: {points.shape}, dtype: {points.dtype}")
print(f"Available fields: {points.dtype.names}")
print(f"Position range X: [{points['px'].min():.2f}, {points['px'].max():.2f}]")
print(f"Confidence range: [{points['inv_dist_std'].min():.4f}, {points['inv_dist_std'].max():.4f}]")

# Filter by confidence threshold
high_conf_points = filter_points_from_confidence(points, min_inv_dist_std=0.001)
print(f"Filtered: {len(points)} → {len(high_conf_points)} points ({100*len(high_conf_points)/len(points):.1f}%)")

In [7]:
#| echo: true
from projectaria_tools.core.sophus import SE3
import numpy as np

# From quaternion + translation
quat_w = 1.0
quat_xyz = [0.0, 0.0, 0.0]
translation = [1.0, 2.0, 3.0]
T = SE3.from_quat_and_translation(quat_w, quat_xyz, translation)
print(f"SE3 from quat+trans:\n{T.to_matrix()}")

# From 4x4 matrix
T_matrix = np.eye(4)
T_matrix[:3, 3] = [1, 2, 3]
T = SE3.from_matrix(T_matrix)
print(f"\nSE3 from matrix:\n{T.to_matrix()}")

# From twist (rotation vector + translation)
rot_vec = [0, 0, 0]
trans_part = [1, 2, 3]
T = SE3.exp(trans_part, rot_vec)
print(f"\nSE3 from exp (twist):\n{T.to_matrix()}")
print(f"Translation part: {T.to_matrix()[:3, 3]}")

SE3 from quat+trans:
[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 3.]
 [0. 0. 0. 1.]]

SE3 from matrix:
[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 3.]
 [0. 0. 0. 1.]]

SE3 from exp (twist):
[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 3.]
 [0. 0. 0. 1.]]
Translation part: [1. 2. 3.]


In [8]:
#| echo: true
# Convert to matrix
T_matrix = T.to_matrix()  # (4, 4) numpy array
T_matrix3x4 = T.to_matrix3x4()  # (3, 4) numpy array
print(f"4x4 matrix shape: {T_matrix.shape}")
print(f"3x4 matrix shape: {T_matrix3x4.shape}")

# Invert transform
T_inv = T.inverse()
identity_check = (T @ T_inv).to_matrix()
print(f"\nInverse check (should be identity):")
print(f"Max deviation from identity: {np.abs(identity_check - np.eye(4)).max():.2e}")

# Compose transforms
T_world_device = SE3.from_quat_and_translation(1.0, [0, 0, 0], [1, 0, 0])
T_device_camera = SE3.from_quat_and_translation(1.0, [0, 0, 0], [0, 0.1, 0])
T_world_camera = T_world_device @ T_device_camera
print(f"\nComposed transform:\n{T_world_camera.to_matrix()}")

# Transform points
points_3d = np.array([[0, 0, 10], [1, 1, 5]])  # (N, 3)
points_transformed = (T @ points_3d.T).T
print(f"\nOriginal points shape: {points_3d.shape}")
print(f"Transformed points shape: {points_transformed.shape}")
print(f"Transformed points:\n{points_transformed}")

4x4 matrix shape: (4, 4)
3x4 matrix shape: (3, 4)

Inverse check (should be identity):
Max deviation from identity: 0.00e+00

Composed transform:
[[1.  0.  0.  1. ]
 [0.  1.  0.  0.1]
 [0.  0.  1.  0. ]
 [0.  0.  0.  1. ]]

Original points shape: (2, 3)
Transformed points shape: (2, 3)
Transformed points:
[[ 1.  2. 13.]
 [ 2.  3.  8.]]


In [9]:
#| echo: true
# Create batch of SE3 transforms
translations = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotations = [[0, 0, 0], [0.1, 0, 0], [0, 0.1, 0]]
T_batch = SE3.exp(translations, rotations)

print(f"Batch size: {len(T_batch)}")
print(f"First transform in batch:\n{T_batch[0].to_matrix()}")

# Apply each transform to a point
point = np.array([0, 0, 10])
print(f"\nOriginal point: {point}")
print(f"Transformed by each SE3 in batch:")
for i, T in enumerate(T_batch):
    transformed_pt = T @ point
    print(f"  T[{i}] @ point = {transformed_pt}")

Batch size: 3
First transform in batch:
[[1. 0. 0. 1.]
 [0. 1. 0. 2.]
 [0. 0. 1. 3.]
 [0. 0. 0. 1.]]

Original point: [ 0  0 10]
Transformed by each SE3 in batch:
  T[0] @ point = [[ 1.]
 [ 2.]
 [13.]]
  T[1] @ point = [[ 4.        ]
 [ 3.69358658]
 [16.18983839]]
  T[2] @ point = [[ 8.43629846]
 [ 8.        ]
 [18.58534072]]


In [10]:
#| echo: true
from projectaria_tools.core.sophus import SO3

# From rotation vector
rot_vec = [0, 0, np.pi/2]  # 90° around Z-axis
R = SO3.exp(rot_vec)
print(f"SO3 from rotation vector (90° around Z):\n{R.to_matrix()}")

# From quaternion (w, [x, y, z])
R = SO3.from_quat(1.0, [0, 0, 0])
print(f"\nSO3 from quaternion (identity):\n{R.to_matrix()}")

# From rotation matrix
R_matrix = np.eye(3)
R = SO3.from_matrix(R_matrix)
print(f"\nSO3 from matrix:\n{R.to_matrix()}")

# Apply to points
points_to_rotate = np.array([[1, 0, 0], [0, 1, 0]])
rotated = (R @ points_to_rotate.T).T
print(f"\nOriginal points:\n{points_to_rotate}")
print(f"Rotated points:\n{rotated}")

SO3 from rotation vector (90° around Z):
[[ 2.22044605e-16 -1.00000000e+00  0.00000000e+00]
 [ 1.00000000e+00  2.22044605e-16  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]

SO3 from quaternion (identity):
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

SO3 from matrix:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Original points:
[[1 0 0]
 [0 1 0]]
Rotated points:
[[1. 0. 0.]
 [0. 1. 0.]]


In [11]:
#| echo: true
from projectaria_tools.projects import ase

# Get ASE RGB camera calibration
camera_calib = ase.get_ase_rgb_calibration()
print(f"Camera calibration type: {type(camera_calib)}")
print(f"Camera model: {camera_calib.get_label()}")

# VRS provider example (requires VRS file):
# device_calib = provider.get_device_calibration()
# camera_calib = device_calib.get_camera_calib("camera-rgb")
# print(f"Available camera labels: {device_calib.get_camera_labels()}")

Camera calibration type: <class '_core_pybinds.calibration.CameraCalibration'>
Camera model: camera-rgb


In [12]:
#| echo: true
# Image dimensions
width, height = camera_calib.get_image_size()
print(f"Image size: {width} x {height}")

# Focal lengths
fx, fy = camera_calib.get_focal_lengths()
print(f"Focal lengths: fx={fx:.2f}, fy={fy:.2f}")

# Principal point
cx, cy = camera_calib.get_principal_point()
print(f"Principal point: cx={cx:.2f}, cy={cy:.2f}")

# Summary
print(f"\nIntrinsics summary:")
print(f"  Resolution: {width}x{height}")
print(f"  FOV (approx): {2 * np.arctan(width/(2*fx)) * 180/np.pi:.1f}° horizontal")

Image size: 704 x 704
Focal lengths: fx=297.64, fy=297.64
Principal point: cx=357.66, cy=349.19

Intrinsics summary:
  Resolution: 704x704
  FOV (approx): 99.6° horizontal


In [13]:
#| echo: true
# Project 3D point in camera frame to pixel
point_in_camera = np.array([0, 0, 10])  # X, Y, Z
pixel = camera_calib.project(point_in_camera)

if pixel is not None:
    u, v = pixel  # Image coordinates
    print(f"Point {point_in_camera} projects to pixel ({u:.2f}, {v:.2f})")
    print(f"Pixel coordinates: u={u:.2f}, v={v:.2f}")
else:
    print("Point projects outside image bounds!")

# Test multiple points
test_points = np.array([[0, 0, 5], [1, 0, 5], [0, 1, 5], [2, 2, 10]])
print(f"\nProjecting {len(test_points)} test points:")
for i, pt in enumerate(test_points):
    px = camera_calib.project(pt)
    if px is not None:
        print(f"  Point {i}: {pt} → pixel [{px[0]:.1f}, {px[1]:.1f}]")
    else:
        print(f"  Point {i}: {pt} → outside image")

Point [ 0  0 10] projects to pixel (357.66, 349.19)
Pixel coordinates: u=357.66, v=349.19

Projecting 4 test points:
  Point 0: [0 0 5] → pixel [357.7, 349.2]
  Point 1: [1 0 5] → pixel [417.3, 349.2]
  Point 2: [0 1 5] → pixel [357.7, 408.8]
  Point 3: [ 2  2 10] → pixel [417.2, 408.7]


In [14]:
#| echo: true
# Unproject pixel to 3D ray (bearing vector)
pixel = [100, 200]
ray = camera_calib.unproject(pixel)

if ray is not None:
    print(f"Pixel {pixel} → ray direction: {ray}")
    print(f"Ray magnitude (should be ~1.0): {np.linalg.norm(ray):.6f}")

    # To get 3D point at depth d:
    depth = 5.0  # meters
    point_3d = depth * ray
    print(f"3D point at depth {depth}m: {point_3d}")
else:
    print("Pixel outside valid image region!")

Pixel [100, 200] → ray direction: [-0.99671047 -0.57624795  1.        ]
Ray magnitude (should be ~1.0): 1.524957
3D point at depth 5.0m: [-4.98355233 -2.88123975  5.        ]


In [15]:
#| echo: true
# Device-to-camera transform
T_device_camera = camera_calib.get_transform_device_camera()
print(f"Device-to-camera transform:\n{T_device_camera.to_matrix()}")

# Use first pose from actual trajectory
T_world_device = Ts_world_device[0]

# Full transform chain: World → Device → Camera
T_camera_world = (
    T_device_camera.inverse() @ T_world_device.inverse()
)
print(f"\nFull transform chain (camera-from-world):\n{T_camera_world.to_matrix()[:3, :]}")

# Project world point to image
point_world = np.array([1, 2, 10])
point_camera = T_camera_world @ point_world
pixel = camera_calib.project(point_camera)
print(f"\nWorld point: {point_world}")
print(f"Camera frame point: {point_camera}")
if pixel is not None:
    print(f"Projected pixel: [{pixel[0]:.2f}, {pixel[1]:.2f}]")
else:
    print("Point not visible in camera")

Device-to-camera transform:
[[ 0.99606003 -0.04388682  0.07706079 -0.0075301 ]
 [ 0.08210934  0.78468796 -0.61442889 -0.01090855]
 [-0.03350334  0.61833547  0.78519983 -0.00359806]
 [ 0.          0.          0.          1.        ]]

Full transform chain (camera-from-world):
[[ 0.03056568 -0.09188739 -0.99530018  1.74678918]
 [ 0.97306464  0.23037144  0.00861464 -2.36902499]
 [ 0.22849716 -0.96875472  0.09645383  3.27943319]]

World point: [ 1  2 10]
Camera frame point: [[-8.35942171]
 [-0.84907107]
 [ 2.53495919]]
Point not visible in camera


In [16]:
#| echo: true
def depth_to_pointcloud(depth_map, camera_calib, T_world_device, subsample=4):
    """
    Convert depth map to 3D point cloud using Aria's native camera model.

    Args:
        depth_map: (H, W) depth in millimeters
        camera_calib: Camera calibration object
        T_world_device: SE3 transform from device to world
        subsample: Downsample factor

    Returns:
        points_world: (N, 3) array in world coordinates
    """
    depth_m = depth_map.astype(np.float32) / 1000.0
    height, width = depth_map.shape
    print(f"Depth map: {depth_map.shape}, range [{depth_map.min()}-{depth_map.max()}] mm")

    # Get transforms
    T_device_camera = camera_calib.get_transform_device_camera()
    T_world_camera = T_world_device @ T_device_camera

    points_world = []
    valid_pixels = 0
    total_pixels = (height // subsample) * (width // subsample)

    for v in range(0, height, subsample):
        for u in range(0, width, subsample):
            # Unproject pixel to ray (handles fisheye distortion!)
            ray = camera_calib.unproject([u, v])

            if ray is not None:
                d = depth_m[v, u]
                if 0 < d <= 10.0:  # Valid depth range
                    # Point in camera frame
                    p_camera = d * ray
                    # Transform to world
                    p_world = T_world_camera @ p_camera
                    points_world.append(p_world)
                    valid_pixels += 1

    points_arr = np.array(points_world)
    print(f"Generated {len(points_arr)} points from {valid_pixels}/{total_pixels} valid pixels")
    if len(points_arr) > 0:
        print(f"Point cloud bounds: X[{points_arr[:, 0].min():.2f}, {points_arr[:, 0].max():.2f}], "
              f"Y[{points_arr[:, 1].min():.2f}, {points_arr[:, 1].max():.2f}], "
              f"Z[{points_arr[:, 2].min():.2f}, {points_arr[:, 2].max():.2f}]")
    return points_arr

In [17]:
#| echo: true
from PIL import Image

print(f"Loading scene from: {DATA_PATH}")

# Data already loaded in section 2.2, reuse it:
# trajectory, Ts_world_device, timestamps, points, entities

print(f"✓ Trajectory: {len(Ts_world_device)} poses")
print(f"✓ Point cloud: {points.shape}")
print(f"✓ Entities: {len(entities)}")

# Get camera calibration
width, height = camera_calib.get_image_size()
print(f"✓ Camera: {width}x{height}")

# Load depth/RGB images for frame 0
frame_idx = 0
frame_id = str(frame_idx).zfill(7)
depth = np.array(Image.open(DATA_PATH / "depth" / f"depth{frame_id}.png"))
rgb = np.array(Image.open(DATA_PATH / "rgb" / f"vignette{frame_id}.jpg"))
print(f"✓ Frame {frame_idx}: RGB {rgb.shape}, Depth {depth.shape}")
print(f"  Depth range: [{depth.min()}, {depth.max()}] mm")
print(f"  RGB dtype: {rgb.dtype}, range: [{rgb.min()}, {rgb.max()}]")

Loading scene from: /home/jandu/repos/NBV/.data/semidense_samples/ase/ase_examples/0
✓ Trajectory: 350 poses
✓ Point cloud: (433426, 3)
✓ Entities: 15
✓ Camera: 704x704
✓ Frame 0: RGB (704, 704, 3), Depth (704, 704)
  Depth range: [815, 4748] mm
  RGB dtype: uint8, range: [0, 192]


In [18]:
#| echo: true
# Get pose for this frame
T_world_device_frame = Ts_world_device[frame_idx]
print(f"Device pose at frame {frame_idx}:")
print(f"  Translation: {T_world_device_frame.to_matrix()[:3, 3]}")

# Transform chain
T_device_camera = camera_calib.get_transform_device_camera()
T_camera_world = T_device_camera.inverse() @ T_world_device_frame.inverse()

# Project subset of points (all points would be too many)
sample_indices = np.linspace(0, len(points)-1, 500, dtype=int)
sampled_points = points[sample_indices]

pixels_list = []
visible_count = 0
for point_world in sampled_points:
    point_camera = T_camera_world @ point_world
    pixel = camera_calib.project(point_camera)
    if pixel is not None:
        # Check if pixel is within image bounds
        if 0 <= pixel[0] < width and 0 <= pixel[1] < height:
            pixels_list.append(pixel)
            visible_count += 1

pixels_arr = np.array(pixels_list) if pixels_list else np.array([]).reshape(0, 2)
print(f"\nProjected {visible_count}/{len(sampled_points)} sampled points visible in frame")
if len(pixels_arr) > 0:
    print(f"Pixel coordinates range: u[{pixels_arr[:, 0].min():.1f}, {pixels_arr[:, 0].max():.1f}], "
          f"v[{pixels_arr[:, 1].min():.1f}, {pixels_arr[:, 1].max():.1f}]")

Device pose at frame 0:
  Translation: [1.51215362 3.88807193 1.43420936]

Projected 148/500 sampled points visible in frame
Pixel coordinates range: u[24.9, 641.9], v[10.6, 685.1]


In [19]:
#| eval: false
#| echo: true
from projectaria_tools.tools.dataset_downloader import (
    DatasetDownloader,
    DatasetDownloadStatusManager
)

# Initialize downloader
sequences = ["sequence_001", "sequence_002"]
data_types = ["trajectory", "depth", "rgb"]
downloader = DatasetDownloader(
    sequences=sequences,
    data_types_selected=data_types,
    overwrite=False
)
print(f"Configured downloader for {len(sequences)} sequences, {len(data_types)} data types")

# Download to folder
output_folder = ".data/ase_dataset"
print(f"Downloading to: {output_folder}")
downloader.download_data(output_folder)

# Track download status
status_manager = DatasetDownloadStatusManager()
status_manager.set_download_status("trajectory", True)
status_manager.to_json("download_status.json")
print(f"Download status saved to download_status.json")

In [20]:
#| eval: false
#| echo: true
from projectaria_tools.tools.aria_rerun_viewer import AriaDataViewer

# Create viewer
viewer = AriaDataViewer()
viewer.set_device_calibration(device_calib)

# Plot images
viewer.plot_image(image_array, label="camera-rgb", device_timestamp_ns=timestamp)

# Plot 3D data
viewer.plot_vio_data(vio_data)
viewer.plot_device_extrinsics()

In [21]:
#| echo: true
print(f"Scene: {DATA_PATH}")
print(f"Total frames: {len(Ts_world_device)}")

# Select views to fuse (every 50th frame for speed)
view_indices = [0, 50, 100, 150, 200]
all_points_list = []

for idx in view_indices:
    # Load depth
    frame_id = str(idx).zfill(7)
    depth_img = np.array(Image.open(DATA_PATH / "depth" / f"depth{frame_id}.png"))

    # Get pose
    T_world_device_view = Ts_world_device[idx]

    # Convert to point cloud
    pc = depth_to_pointcloud(
        depth_img, camera_calib, T_world_device_view, subsample=8
    )
    all_points_list.append(pc)
    print(f"  View {idx}: {len(pc):,} points")

# Fuse all views
fused_pc = np.vstack(all_points_list)
print(f"\n✓ Fused point cloud: {len(fused_pc):,} points from {len(view_indices)} views")
print(f"  Bounds: X[{fused_pc[:, 0].min():.2f}, {fused_pc[:, 0].max():.2f}], "
      f"Y[{fused_pc[:, 1].min():.2f}, {fused_pc[:, 1].max():.2f}], "
      f"Z[{fused_pc[:, 2].min():.2f}, {fused_pc[:, 2].max():.2f}]")

Scene: /home/jandu/repos/NBV/.data/semidense_samples/ase/ase_examples/0
Total frames: 350
Depth map: (704, 704), range [815-4748] mm
Generated 6121 points from 6121/7744 valid pixels
Point cloud bounds: X[-2.28, 8.33], Y[0.13, 3.02], Z[-1.26, 4.93]
  View 0: 6,121 points
Depth map: (704, 704), range [1330-6974] mm


Generated 6121 points from 6121/7744 valid pixels


Point cloud bounds: X[-4.74, 2.86], Y[0.05, 8.14], Z[-1.47, 4.74]
  View 50: 6,121 points


Depth map: (704, 704), range [1262-65535] mm


Generated 6121 points from 6121/7744 valid pixels
Point cloud bounds: X[-0.16, 6.60], Y[0.95, 9.05], Z[-1.39, 4.82]
  View 100: 6,121 points


Depth map: (704, 704), range [1135-7957] mm


Generated 6121 points from 6121/7744 valid pixels


Point cloud bounds: X[-8.33, 5.24], Y[-0.97, 7.11], Z[-1.15, 5.09]
  View 150: 6,121 points
Depth map: (704, 704), range [1041-5922] mm


Generated 6121 points from 6121/7744 valid pixels
Point cloud bounds: X[-3.65, 2.16], Y[-7.98, 2.14], Z[-1.13, 5.09]
  View 200: 6,121 points

✓ Fused point cloud: 30,605 points from 5 views
  Bounds: X[-8.33, 8.33], Y[-7.98, 9.05], Z[-1.47, 5.09]


In [22]:
#| eval: false
#| echo: true
# VRS data
provider = data_provider.create_vrs_data_provider(vrs_path)
image_data, record = provider.get_image_data_by_index(stream_id, idx)

# ASE data
trajectory = readers.read_trajectory_file(traj_csv)
points = readers.read_points_file(points_csv_gz)
entities = readers.read_language_file(language_txt)

In [23]:
#| eval: false
#| echo: true
# SE3 creation
T = SE3.from_quat_and_translation(w, xyz, translation)
T = SE3.from_matrix(T_4x4)

# SE3 operations
T_inv = T.inverse()
T_composed = T1 @ T2
points_transformed = T @ points_3d

In [24]:
#| eval: false
#| echo: true
# Projection
pixel = camera_calib.project(point_in_camera)

# Unprojection
ray = camera_calib.unproject(pixel)
point_3d = depth * ray

# Extrinsics
T_device_camera = camera_calib.get_transform_device_camera()

In [25]:
#| eval: false
#| echo: true
# This assumes pinhole model - WRONG for Aria!
intrinsic = o3d.camera.PinholeCameraIntrinsic(width, height, fx, fy, cx, cy)
pcd = o3d.geometry.PointCloud.create_from_depth_image(depth, intrinsic)

In [26]:
#| eval: false
#| echo: true
# Handles fisheye distortion correctly
ray = camera_calib.unproject([u, v])
point_3d = depth * ray

In [27]:
#| eval: false
#| echo: true
# Point is in world frame, but project() expects camera frame!
pixel = camera_calib.project(point_world)  # WRONG

In [28]:
#| eval: false
#| echo: true
point_camera = T_camera_world @ point_world
pixel = camera_calib.project(point_camera)  # CORRECT